# Whisper-small H70 live wake-word inference

From the GPTWake repository, open this notebook with your existing GPTWake Jupyter kernel. Run the two setup cells, wait for the model to load, and then run **Listen**. The first run downloads the matching Whisper-small backbone from Hugging Face. The trained head is included on this branch.

If an import is missing, add the project's `xlsr` optional dependencies to the existing environment. The pilot runs on CPU. The microphone uses the system default unless you set `INPUT_DEVICE` to a device index from `sd.query_devices()`.

The first prediction uses 2.5 seconds of audio. After that, the notebook scores a rolling window every 200 ms. It prints `0` while listening and prints `1` once before closing the stream. Set `WAKE_THRESHOLD` in **Listen** to any value from 0 to 1 and rerun that cell to apply it. Higher values make detection stricter. Use **Interrupt Kernel** to stop listening manually.

In [ ]:
from pathlib import Path

import numpy as np
import torch
from huggingface_hub import hf_hub_download
from safetensors import safe_open
from torch import nn
from transformers import WhisperConfig, WhisperFeatureExtractor
from transformers.models.whisper.modeling_whisper import WhisperEncoder

SAMPLE_RATE = 16_000
WINDOW_SAMPLES = 40_000
STRIDE_SECONDS = 0.20
CPU_THREADS = 4
INPUT_DEVICE = None

# Sigmoid of the epoch-15 logit cutoff at 5/100 validation false accepts.
DEFAULT_WAKE_THRESHOLD = 0.7811700052684897
CHECKPOINT_PATH = Path("checkpoints/whisper_h70_pilot.pt")
DEVICE = torch.device("cpu")
torch.set_num_threads(CPU_THREADS)

project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents) if (folder / CHECKPOINT_PATH).is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError(f"Start Jupyter from the GPTWake checkout containing {CHECKPOINT_PATH}.")

checkpoint = torch.load(project_root / CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
if (checkpoint["model_id"], checkpoint["preset"], checkpoint["pooling"], checkpoint["epoch"]) != (
    "openai/whisper-small", "H70", "attention", 15
):
    raise ValueError("The validation threshold belongs to the Whisper-small H70 attention checkpoint at epoch 15.")

model_id, revision = checkpoint["model_id"], checkpoint["revision"]
extractor = WhisperFeatureExtractor.from_pretrained(model_id, revision=revision)
config = WhisperConfig.from_pretrained(model_id, revision=revision)
config.max_source_positions = checkpoint["max_source_positions"]
encoder = WhisperEncoder(config)

weights_path = hf_hub_download(model_id, "model.safetensors", revision=revision)
prefix = "model.encoder."
with safe_open(weights_path, framework="pt", device="cpu") as weights:
    encoder_state = {
        key.removeprefix(prefix): weights.get_tensor(key)
        for key in weights.keys()
        if key.startswith(prefix)
    }
    encoder_state["embed_positions.weight"] = encoder_state["embed_positions.weight"][
        :config.max_source_positions
    ]
    encoder.load_state_dict(encoder_state, strict=True)
del encoder_state
encoder.to(DEVICE).eval().requires_grad_(False)


class AttentionHead(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.classifier = nn.Linear(width, 1)
        self.scorer = nn.Linear(width, 1, bias=False)

    def forward(self, tokens):
        weights = self.scorer(tokens).softmax(dim=1)
        pooled = torch.bmm(weights.transpose(1, 2), tokens).squeeze(1)
        return self.classifier(pooled)


head = AttentionHead(config.d_model)
head.load_state_dict(checkpoint["head"], strict=True)
head.to(DEVICE).eval().requires_grad_(False)


@torch.inference_mode()
def score_window(window):
    """Return the 0-to-1 wake score for one mono 16 kHz, 2.5-second window."""
    window = np.ascontiguousarray(window, dtype=np.float32)
    if window.shape != (WINDOW_SAMPLES,):
        raise ValueError(f"Expected {WINDOW_SAMPLES} mono samples, got {window.shape}.")
    mel = extractor(
        window[None, :],
        sampling_rate=SAMPLE_RATE,
        padding=False,
        truncation=False,
        return_tensors="pt",
    ).input_features
    logit = head(encoder(mel).last_hidden_state)
    return logit.double().sigmoid().item()


# Warm up the CPU before opening the microphone.
score_window(np.zeros(WINDOW_SAMPLES, dtype=np.float32))
print(f"Loaded Whisper H70 epoch {checkpoint['epoch']} on CPU. Default threshold {DEFAULT_WAKE_THRESHOLD:.6f}.")

In [ ]:
import math

import sounddevice as sd
from scipy.signal import resample_poly


def microphone_rate(device):
    """Prefer direct 16 kHz capture and fall back to the device's default rate."""
    try:
        sd.check_input_settings(device=device, samplerate=SAMPLE_RATE, channels=1, dtype="float32")
        return SAMPLE_RATE
    except sd.PortAudioError:
        rate = round(sd.query_devices(device, "input")["default_samplerate"])
        sd.check_input_settings(device=device, samplerate=rate, channels=1, dtype="float32")
        return rate


def listen_to_stream(stream, capture_rate, *, threshold, score=score_window):
    """Print one decision per rolling window and return on the first detection."""
    capture_window = round(WINDOW_SAMPLES * capture_rate / SAMPLE_RATE)
    capture_hop = round(STRIDE_SECONDS * capture_rate)
    if not 1 <= capture_hop <= capture_window:
        raise ValueError("Set STRIDE_SECONDS to a positive interval of at most 2.5 seconds.")

    common = math.gcd(SAMPLE_RATE, capture_rate)
    window, overflow = stream.read(capture_window)
    while True:
        if overflow:
            # Restart the window after dropped input so it contains contiguous audio.
            window, overflow = stream.read(capture_window)
            continue
        mono = window[:, 0]
        if capture_rate != SAMPLE_RATE:
            mono = resample_poly(mono, SAMPLE_RATE // common, capture_rate // common)
            mono = mono[:WINDOW_SAMPLES]
        predicted = int(score(mono) >= threshold)
        print(predicted, end="\n" if predicted else " ", flush=True)
        if predicted:
            return
        frames, overflow = stream.read(capture_hop)
        window = np.concatenate((window[capture_hop:], frames), axis=0)


def listen(*, threshold, device=INPUT_DEVICE):
    if not 0.0 <= threshold <= 1.0:
        raise ValueError("WAKE_THRESHOLD must be between 0 and 1.")
    rate = microphone_rate(device)
    with sd.InputStream(device=device, samplerate=rate, channels=1, dtype="float32") as stream:
        listen_to_stream(stream, rate, threshold=threshold)

In [ ]:
# Listen
WAKE_THRESHOLD = DEFAULT_WAKE_THRESHOLD  # Replace with a value from 0.0 to 1.0 to tune detection.
try:
    listen(threshold=WAKE_THRESHOLD)
except KeyboardInterrupt:
    print("\nMicrophone stopped.")